# The jet wake in FastHydro

$\Delta e = e(\text{jet}) - e(\text{no jet})$ for a shower produced by X-SCAPE's **real**
energy-loss chain — Matter + LBT feeding the CausalLiquefier — evolved on the fast 3+1D Milne
solver.

**One file per leg.** FastHydro runs the background and the jet in the same event, from one
initial condition, and writes both into a single HDF5 file (`arr` = jet, `arr_bg` = background).
So `jet − nojet` needs no pairing of separate runs and cannot be paired wrongly: the writer
refuses to store a pair whose two legs did not share an IC. FNO4d's `viscous_vs_ideal.ipynb`,
which this is adapted from, needs four files for the same two comparisons; here it is two.

| | file |
|---|---|
| **ideal** | `out_wake/wake_ideal.h5` |
| **viscous** ($\eta/s = 0.08$) | `out_wake/wake_visc.h5` |

Make them with **`example/make_wake_data.py`** (§1 prints the command). Central Au+Au 200 GeV
on the 65×65×33 grid at 0.3125 fm with the hotQCD/SMASH lattice EoS — the same medium FNO4d's
`config_AuAu200_central_jet.yaml` was measured on.

**One control needs care.** The shower responds to the medium it traverses, so running
Matter+LBT twice — once against an ideal background, once against a viscous one — gives *two
different droplet sets*. That is real physics, but it would mean `visc − ideal` mixes the
hydrodynamic response with a different jet. So the generator runs the first leg live and
**replays its droplets** through the other solver; §2 checks that the two droplet tables really
are identical. (`make_wake_data.py --live` gives the uncontrolled version, which is the more
complete physical statement and the less interpretable comparison.)

> **Scope.** One event, one $\eta/s$, shear only (bulk is off and untested upstream). This is
> about the *sign and mechanism* of the viscous effect on a wake, not its calibrated size. The
> front width in §5 is a coarse measure quantised by the 0.3125 fm cell — read it as an
> ordering, not a measurement.

In [ ]:
import os, subprocess, sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

# the contribution's own python/ , so this runs from a checkout with no install
HERE = os.path.dirname(os.path.abspath("__file__" in dir() and __file__ or "."))
for p in (os.path.join("..", "python"), os.path.join(HERE, "..", "python")):
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from fasthydro.browse import PairBrowser

OUT = os.environ.get("WAKE_OUT", "out_wake")     # gitignored

# --- one fixed visual language --------------------------------------------------------------
# Two series, so the hues are ASSIGNED, never cycled: ideal is always blue, viscous always
# vermillion, in every panel, and each is reinforced with a line style so the pair survives
# colour-vision deficiency and greyscale. Colour never carries identity alone.
C_ID, C_VI = "#0072B2", "#D55E00"
STYLE = {"ideal": dict(color=C_ID, ls="-",  lw=1.9, label="ideal"),
         "visc":  dict(color=C_VI, ls="--", lw=1.9, label=r"viscous  $\eta/s=0.08$")}
CMAP_DIFF = "RdBu_r"        # diverging, about a neutral midpoint, for signed de
CMAP_E    = "inferno"       # sequential, for e itself

def tidy(ax, xlabel=None, ylabel=None, title=None):
    """Recessive grid and spines, so the data is the most prominent thing in the frame."""
    ax.grid(True, alpha=0.25, lw=0.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    if title:  ax.set_title(title, fontsize=11)
    return ax

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.titlesize": 11,
                     "legend.frameon": False})
print("ready")

## 1. The data

Two runs, differing in one switch. If the files are not there, the cell prints the commands
that make them rather than running a half-hour job behind your back.

In [ ]:
LEGS = {"ideal": "ideal", "visc": "israel_stewart"}
PATH = {k: os.path.join(OUT, f"wake_{k}.h5") for k in LEGS}

missing = [k for k, p in PATH.items() if not os.path.exists(p)]
if missing:
    print("Missing:", ", ".join(PATH[k] for k in missing))
    print("\nGenerate both legs from your X-SCAPE build tree:\n")
    print("    python ../external_packages/js-contrib/contribs/FastHydro/example/"
          "make_wake_data.py\n")
    print("Then either run this notebook from that build tree, or point it at the output:\n")
    print(f"    WAKE_OUT=<build>/out_wake jupyter lab jet_wake.ipynb\n")
    print("It takes about a minute per leg on MPS. --device cpu is slower and bitwise "
          "reproducible;\n--live runs each leg end to end instead of replaying the droplets "
          "(see the note above).")
    raise SystemExit("generate the files above, then re-run this cell")

P = {k: PairBrowser(v) for k, v in PATH.items()}
for k, p in P.items():
    print(f"{k:6s} {p}")

TAU = P["ideal"].tau
x, y = P["ideal"].x, P["ideal"].y
X, Y = np.meshgrid(x, y, indexing="ij")
ie0 = P["ideal"].neta // 2
print(f"\ntau grid: {TAU[0]:.2f} .. {TAU[-1]:.2f} fm/c in {len(TAU)} frames")

## 2. Controls

Nothing below means anything unless these hold. Each is a property the *pairing* must have,
not a property of the physics:

1. both legs of a file start from the same initial condition — otherwise `jet − nojet` is not
   the jet;
2. the two files deposit the same droplets — otherwise `visc − ideal` is not the viscosity;
3. the legs are bit-identical until the first deposit — a difference before the jet has
   deposited anything would mean something else is leaking in.

In [ ]:
ok = True
print("(1) both legs share an initial condition")
for k, p in P.items():
    good = p.ic_identical(0)
    ok &= good
    print(f"      {k:6s} {good}")

print("\n(2) the same deposit in both files")
dep = {k: p.jet.droplets(0) for k, p in P.items()}
for k, d in dep.items():
    print(f"      {k:6s} {len(d):3d} droplets, E_dep = {d[:, 4].sum():7.3f} GeV")
same_drops = np.allclose(dep["ideal"], dep["visc"])
ok &= same_drops
print(f"      identical droplet tables: {same_drops}")

print("\n(3) bit-identical until the jet deposits")
t_dep0 = float(dep["ideal"][:, 0].min()) + float(P["ideal"].attrs["liquefier_tau_delay"])
for k, p in P.items():
    first = p.first_difference(0)
    print(f"      {k:6s} first differing frame {first} (tau = {TAU[first]:.2f} fm/c); "
          f"first deposit at tau = {t_dep0:.2f}")

print("\n(4) freeze-out [fm/c]")
for k, p in P.items():
    print(f"      {k:6s} background {p.bg.tau_fo[0]:5.2f}   jet {p.jet.tau_fo[0]:5.2f}   "
          f"frames where both are live: {p.live(0)}")

print(f"\ncontrols pass: {bool(ok)}")

## 3. The wake in the $\eta = 0$ plane

Red is compression — the Mach front — blue the depletion trailing it, the diffusion wake. The
star is where the jet was at that $\tau$, read from the droplet table rather than reconstructed
from the config, so it is right even for a jet that starts off-centre.

**One plotting decision does all the work.** Both rows share a single symmetric colour scale
per column, and that scale *excludes a disc of twice the deposition radius around the source*.
Letting each panel pick its own scale would hide exactly the effect we are after; scaling to
the global maximum would put the scale on the cell currently being deposited into, which runs
an order of magnitude above the wake and leaves the cone invisible.

In [ ]:
R_EXCL = 2.0 * P["ideal"].blob_radius()
live = min(P[k].live(0) for k in LEGS)
FRAMES = [int(round(f)) for f in np.linspace(live * 0.35, live * 0.95, 4)]

def wake_scale(k, itau):
    """max |de| OUTSIDE the deposition blob -- the wake's own scale, not the source's."""
    de = P[k].diff(0, itau)
    pos = P[k].source_at(0, TAU[itau])
    if pos is None:
        return float(np.abs(de).max())
    far = ((X - pos[0])**2 + (Y - pos[1])**2) > R_EXCL**2
    return float(np.abs(de[far, :]).max()) if far.any() else 0.0

VMAX = {i: max(wake_scale("ideal", i), wake_scale("visc", i), 1e-12) for i in FRAMES}

fig, axes = plt.subplots(2, len(FRAMES), figsize=(3.6*len(FRAMES), 6.6),
                         sharex=True, sharey=True, constrained_layout=True)
for row, k in enumerate(("ideal", "visc")):
    for col, itau in enumerate(FRAMES):
        ax = axes[row, col]
        de = P[k].diff(0, itau)[:, :, ie0]
        v = VMAX[itau]
        ax.pcolormesh(x, y, de.T, cmap=CMAP_DIFF, vmin=-v, vmax=v, shading="auto")
        pos = P[k].source_at(0, TAU[itau])
        if pos is not None:
            ax.plot([pos[0]], [pos[1]], "k*", ms=12, mec="w", mew=0.8)
        ax.set_aspect("equal")
        if row == 0: ax.set_title(rf"$\tau$ = {TAU[itau]:.1f} fm", fontsize=10)
        if col == 0: ax.set_ylabel(f"{STYLE[k]['label']}\n\ny [fm]", fontsize=9)
        if row == 1: ax.set_xlabel("x [fm]")
# one bar per COLUMN, spanning both rows: the two rows share that column's scale, which is the
# whole point. A bar on every panel would just repeat it.
for col, itau in enumerate(FRAMES):
    cb = fig.colorbar(axes[0, col].collections[0], ax=axes[:, col], fraction=0.05, pad=0.02)
    cb.ax.tick_params(labelsize=7)
    if col == len(FRAMES) - 1:
        cb.set_label(r"$\Delta e$ [GeV/fm$^3$]", fontsize=9)
fig.suptitle(r"Jet wake at $\eta=0$ — each column shares one colour scale between rows",
             fontsize=11)

## 4. The same wake in $\eta$ and $\varphi$

The $x$–$y$ maps show where the energy is; this shows how it is distributed in the two angles a
jet measurement bins in. Each cell's excess $\Delta e$ is weighted by its proper volume
$\tau\,\mathrm{d}x\,\mathrm{d}y\,\mathrm{d}\eta$ and summed into $(\eta, \varphi)$ bins, giving
$\mathrm{d}\Delta E / \mathrm{d}\eta\,\mathrm{d}\varphi$ in GeV.

> **These are SPATIAL angles, not the $(\eta, \varphi)$ of hadrons.** $\eta$ here is the
> spacetime rapidity $\eta_s$ of the fluid cell and $\varphi = \arctan(y/x)$ its azimuth in the
> transverse plane. Turning this into the momentum-space $(\eta, \varphi)$ a detector reports
> would need a Cooper–Frye surface, which FastHydro does not compute. Read these as *where the
> deposited energy sits in the fireball*, which is what they are.

In [ ]:
NPHI = 24
phi_edges = np.linspace(-np.pi, np.pi, NPHI + 1)
phi_cen = 0.5 * (phi_edges[:-1] + phi_edges[1:])

_a = P["ideal"].attrs
DX, DY, DETA = (float(_a["dx"]), float(_a["dy"]), float(_a["deta"]))
eta_edges = np.concatenate([P["ideal"].eta - 0.5 * DETA, [P["ideal"].eta[-1] + 0.5 * DETA]])
PHI_XY = np.arctan2(Y, X)                      # (nx, ny) azimuth of each transverse cell


def eta_phi(k, itau, *, dphi=0.0, deta_shift=0.0):
    '''-> (neta, NPHI) array of dE/deta dphi [GeV], optionally re-centred on the jet.'''
    p = P[k]
    de = p.diff(0, itau)                                     # (nx, ny, neta)
    w = de * p.tau[itau] * DX * DY * DETA                    # GeV in each cell
    phi = np.mod(PHI_XY[:, :, None] - dphi + np.pi, 2 * np.pi) - np.pi
    eta = np.broadcast_to(p.eta[None, None, :] - deta_shift, de.shape)
    H, _, _ = np.histogram2d(eta.ravel(), np.broadcast_to(phi, de.shape).ravel(),
                             bins=[eta_edges, phi_edges], weights=w.ravel())
    return H / (DETA * (2 * np.pi / NPHI))                   # per unit eta, per unit phi


# The jet was produced at phi ~ -141 deg, so on a plain [-pi, pi) axis its wake straddles the
# wrap and appears as two pieces at opposite edges. Keep phi ABSOLUTE but slide the 2*pi
# window so it is centred on the production azimuth; the ticks carry the real values.
_d0 = P["ideal"].jet.droplets(0)[0]
PHI0 = float(np.arctan2(_d0[2], _d0[1]))
_wrap = lambda a: (a + np.pi) % (2 * np.pi) - np.pi
_tick = lambda off: f"{np.degrees(_wrap(PHI0 + off)):+.0f}" + r"$^\circ$"

fig, axes = plt.subplots(2, len(FRAMES), figsize=(3.6 * len(FRAMES), 6.2),
                         sharex=True, sharey=True, constrained_layout=True)
VM = {i: max(np.abs(eta_phi(k, i, dphi=PHI0)).max() for k in LEGS) for i in FRAMES}
for row, k in enumerate(("ideal", "visc")):
    for col, itau in enumerate(FRAMES):
        ax, v = axes[row, col], max(VM[itau], 1e-12)
        ax.pcolormesh(phi_edges, eta_edges, eta_phi(k, itau, dphi=PHI0),
                      cmap=CMAP_DIFF, vmin=-v, vmax=v, shading="auto")
        ax.set_ylim(-3, 3)
        # quarter-turn ticks: the two EDGES are the same angle (the window wraps), so
        # labelling both would print one value twice and look like a mistake.
        ax.set_xticks([-np.pi / 2, 0, np.pi / 2])
        ax.set_xticklabels([_tick(-np.pi / 2), _tick(0.0), _tick(np.pi / 2)], fontsize=8)
        if row == 0: ax.set_title(rf"$\tau$ = {TAU[itau]:.1f} fm", fontsize=10)
        if col == 0: ax.set_ylabel(f"{STYLE[k]['label']}\n\n" + r"$\eta_s$", fontsize=9)
        if row == 1: ax.set_xlabel(r"$\varphi$  (about the beam axis)")
for col, itau in enumerate(FRAMES):
    cb = fig.colorbar(axes[0, col].collections[0], ax=axes[:, col], fraction=0.05, pad=0.02)
    cb.ax.tick_params(labelsize=7)
    if col == len(FRAMES) - 1:
        cb.set_label(r"$d\Delta E/d\eta\,d\varphi$  [GeV]", fontsize=9)
fig.suptitle(r"Wake in spatial $(\eta_s, \varphi)$ — each column shares a scale between rows"
             "\n" + rf"$\varphi$ window centred on the jet's production azimuth "
             rf"({np.degrees(PHI0):+.0f}$^\circ$) so the lobe is not split by the wrap",
             fontsize=10)

### Re-centred on the jet

The jet is not at the middle of the fireball — this one was produced 4.6 fm out, at
$(x,y) = (-3.6, -2.9)$, because the vertex is sampled from the binary-collision density. So
$\varphi$ measured about the *beam axis*, as above, is not a jet-relative angle: subtracting the
jet's momentum azimuth from a cell's position azimuth about the origin only means something if
the jet started at the origin, and the offset would dominate the result.

Here $\varphi$ is instead measured **about the jet's own position at that $\tau$**, taken from
the droplet table by `source_at`, with $\Delta\varphi = 0$ pointing along the direction the jet
is travelling. That is the apex of the cone, so this is the same variable §6's Mach angle is
about: a Mach cone appears as two lobes at $\pm\theta_M$, and the diffusion wake as the
depletion between them at $\Delta\varphi \approx 0$.

Once the source stops the apex is pinned to the last deposit and the pattern opens up, as §6
notes — so read the later frames as "expanding away from where the jet stopped", not as a cone.

In [ ]:
def jet_direction_phi(k, event=0):
    '''Azimuth of the summed deposited momentum: the direction the jet was travelling.'''
    d = P[k].jet.droplets(event)
    p3 = d[:, 5:8].sum(axis=0)
    return float(np.arctan2(p3[1], p3[0]))


dphi_edges = np.linspace(0.0, 2 * np.pi, NPHI + 1)
dphi_cen = 0.5 * (dphi_edges[:-1] + dphi_edges[1:])

PHI_DIR = {k: jet_direction_phi(k) for k in LEGS}
for k, ph in PHI_DIR.items():
    d = P[k].jet.droplets(0)
    print(f"{k:6s} travelling at phi = {np.degrees(ph):+7.1f} deg, "
          f"produced at (x, y) = ({d[0, 1]:+.2f}, {d[0, 2]:+.2f}) fm, "
          f"{np.hypot(d[0, 1], d[0, 2]):.2f} fm from the centre")


def eta_phi_jet(k, itau):
    '''dE/deta dphi about the jet's own position at this tau; dphi = 0 is downstream.'''
    p = P[k]
    apex = p.source_at(0, p.tau[itau])
    if apex is None:
        return None
    de = p.diff(0, itau)
    w = de * p.tau[itau] * DX * DY * DETA
    # wrapped into [0, 2pi): 0 = along the direction of travel, 180 deg = directly behind,
    # which is where a trailing wake sits -- putting it in the middle rather than split
    # across both edges.
    phi = np.arctan2(Y - apex[1], X - apex[0])[:, :, None] - PHI_DIR[k]
    phi = np.mod(phi, 2 * np.pi)
    eta = np.broadcast_to(p.eta[None, None, :] - apex[2], de.shape)
    H, _, _ = np.histogram2d(eta.ravel(), np.broadcast_to(phi, de.shape).ravel(),
                             bins=[eta_edges, dphi_edges], weights=w.ravel())
    return H / (DETA * (2 * np.pi / NPHI))


# Show a frame where the jet is STILL DEPOSITING. Once it stops, the apex is pinned to the
# last deposit and what expands away from it is no longer a cone, so a late frame would be a
# misleading thing to put under a "re-centred on the jet" heading.
live_n = min(P[k].live(0) for k in LEGS)
active = [i for i in range(live_n) if all(P[k].source_active(0, TAU[i]) for k in LEGS)]
itau = active[int(0.8 * len(active))] if active else FRAMES[-2]
print(f"frame {itau} (tau = {TAU[itau]:.2f} fm/c); jet still depositing: "
      f"{P['ideal'].source_active(0, TAU[itau])}"
      + ("" if active else "  [no frame has it live; falling back]"))

maps = {k: eta_phi_jet(k, itau) for k in LEGS}

fig = plt.figure(figsize=(13.5, 4.0), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.15])
v = max(max(np.abs(m).max() for m in maps.values()), 1e-12)
for col, k in enumerate(("ideal", "visc")):
    ax = fig.add_subplot(gs[0, col])
    im = ax.pcolormesh(dphi_edges, eta_edges, maps[k], cmap=CMAP_DIFF,
                       vmin=-v, vmax=v, shading="auto")
    ax.axvline(np.pi, color="0.25", lw=0.8, ls=":")
    ax.set_ylim(-3, 3)
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
    ax.set_xticklabels(["0", "90", "180", "270", "360"], fontsize=8)
    ax.set_xlabel(r"$\Delta\varphi$ from the jet direction [deg]"
                  "\n0 = ahead,  180 = behind")
    ax.set_ylabel(r"$\Delta\eta$ from the jet" if col == 0 else "")
    ax.set_title(f"{STYLE[k]['label']}" + rf",  $\tau$ = {TAU[itau]:.1f} fm", fontsize=10)
fig.colorbar(im, ax=fig.axes[:2], fraction=0.04, pad=0.02,
             label=r"$d\Delta E/d\eta\,d\varphi$  [GeV]")

ax = fig.add_subplot(gs[0, 2])
band = np.abs(0.5 * (eta_edges[:-1] + eta_edges[1:])) < 1.0
for k in LEGS:
    ax.plot(np.degrees(dphi_cen), maps[k][band].sum(axis=0) * DETA, **STYLE[k])
ax.axvline(180.0, color="0.25", lw=0.8, ls=":")
ax.set_xticks([0, 90, 180, 270, 360])
tidy(ax, r"$\Delta\varphi$ from the jet [deg]   (0 = ahead, 180 = behind)",
     r"$d\Delta E/d\varphi$  [GeV]", r"projected over $|\Delta\eta| < 1$")
ax.legend(fontsize=9)

print()
for k in LEGS:
    prof = maps[k][band].sum(axis=0) * DETA
    fwd = np.minimum(dphi_cen, 2 * np.pi - dphi_cen) < np.pi / 4
    print(f"{k:6s} within 45 deg of the jet direction: "
          f"{prof[fwd].sum() * (2 * np.pi / NPHI):+.3f} GeV; "
          f"peak at {np.degrees(dphi_cen[np.argmax(prof)]):.0f} deg; "
          f"total |dE/dphi| = {np.abs(prof).sum() * (2 * np.pi / NPHI):.3f} GeV")

## 5. How big, how much, how wide

Three reductions of the same difference, all excluding the deposition blob so they describe the
*wake* and not the source:

- **amplitude** — $\max|\Delta e|$ outside the blob;
- **total disturbance** — $\int|\Delta e|\,\mathrm{d}V$, which counts the depletion as well as
  the front;
- **front width** — the extent of the positive ridge on a cut 2 fm behind the source.

The expectation, stated before the numbers are read: shear viscosity should **damp** the
amplitude and **broaden** the front.

In [ ]:
def wake_series(k):
    p = P[k]
    n = p.live(0)
    amp, tot, wid = np.full(n, np.nan), np.zeros(n), np.full(n, np.nan)
    for i in range(n):
        de = p.diff(0, i)
        pos = p.source_at(0, TAU[i])
        far = np.ones(de.shape[:2], bool)
        if pos is not None:
            far = ((X - pos[0])**2 + (Y - pos[1])**2) > R_EXCL**2
        if far.any():
            amp[i] = float(np.abs(de[far, :]).max())
        tot[i] = p.wake_energy(0, i)
        if pos is not None:                       # front width 2 fm behind the source
            ix = int(np.argmin(np.abs(x - (pos[0] - 2.0))))
            prof = de[ix, :, ie0]
            if prof.max() > 0:
                m = prof > 0.2 * prof.max()
                if m.any():
                    wid[i] = float(y[m].max() - y[m].min())
    return amp, tot, wid

W = {k: wake_series(k) for k in LEGS}
n = min(len(W[k][0]) for k in LEGS)
t = TAU[:n]

fig, axes = plt.subplots(1, 3, figsize=(14.0, 3.8))
for k in LEGS:
    a, E, w = (v[:n] for v in W[k])
    axes[0].plot(t, a, **STYLE[k])
    axes[1].plot(t, E, **STYLE[k])
    axes[2].plot(t, w, **STYLE[k])
tidy(axes[0], r"$\tau$ [fm/c]", r"max $|\Delta e|$ outside the blob [GeV/fm$^3$]",
     "Wake amplitude")
tidy(axes[1], r"$\tau$ [fm/c]", r"$\int|\Delta e|\,dV$ [GeV]", "Total disturbance")
tidy(axes[2], r"$\tau$ [fm/c]", "front width [fm]", "Front width")
axes[2].text(0.03, 0.95, f"steps = grid spacing {float(P['ideal'].attrs['dy']):.2f} fm",
             transform=axes[2].transAxes, fontsize=7, color="0.45", va="top")
for ax in axes: ax.legend(fontsize=9)
fig.tight_layout()

late = t > t[len(t)//2]
def ratio(j):
    a, b = W["visc"][j][:n][late], W["ideal"][j][:n][late]
    m = np.isfinite(a) & np.isfinite(b) & (np.abs(b) > 0)
    return float(np.mean(a[m]/b[m])) if m.any() else np.nan
def delta(j):
    a, b = W["visc"][j][:n][late], W["ideal"][j][:n][late]
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.mean(a[m]-b[m])) if m.any() else np.nan

print(f"over the second half of the live window (tau > {t[len(t)//2]:.1f} fm/c):")
print(f"  amplitude   viscous/ideal : {ratio(0):.3f}   (expect < 1: damping)")
print(f"  total |de|  viscous/ideal : {ratio(1):.3f}")
print(f"  front width viscous-ideal : {delta(2):+.2f} fm  (expect > 0: broadening)")

## 6. Is the front where a Mach cone should be?

`mach_angle` returns $\arcsin(c_s/v)$ with $v=1$, evaluated on the **background** medium at the
source. That is a lower bound on what you can measure: the fireball is expanding, and the
transverse flow at the front adds to $c_s$ and opens the cone. Treat a measured angle *above*
this as consistent, not as a discrepancy.

In [ ]:
print(f"{'tau':>6}  {'c_s at source':>14}  {'Mach half-angle':>16}   source still depositing?")
for itau in FRAMES:
    ang = P["ideal"].mach_angle(0, itau)
    pos = P["ideal"].source_at(0, TAU[itau])
    cs = None
    if pos is not None:
        ix = int(np.argmin(np.abs(x - pos[0]))); iy = int(np.argmin(np.abs(y - pos[1])))
        ie = int(np.argmin(np.abs(P["ideal"].eta - pos[2])))
        cs = P["ideal"].sound_speed(float(P["ideal"].bg.frame(0, itau, 0)[ix, iy, ie]))
    active = P["ideal"].source_active(0, TAU[itau])
    print(f"{TAU[itau]:6.2f}  {cs if cs is None else f'{cs:14.4f}'}  "
          f"{'n/a' if ang is None else f'{ang:13.1f} deg'}   {active}")
print("\nOnce the source stops, the apex is pinned to the last deposit and the front simply")
print("expands away from it, so the opening angle keeps growing and is no longer a Mach angle.")

## 7. Freeze-out — the jet extends the lifetime

The deposited energy has to go somewhere. It reheats the medium, so the jet leg stays above
$T_\text{fo}$ longer than its own background. The effect is small and it is *not* a numerical
artefact: both legs are the same solver on the same initial condition, differing only in the
source.

In [ ]:
print(f"{'leg':8s} {'tau_fo bg':>10} {'tau_fo jet':>11} {'delta':>8}   {'live frames':>11}")
for k, p in P.items():
    b, j = float(p.bg.tau_fo[0]), float(p.jet.tau_fo[0])
    print(f"{STYLE[k]['label'][:8]:8s} {b:10.2f} {j:11.2f} {j-b:+8.2f}   {p.live(0):11d}")
print(f"\ndeposited: {P['ideal'].deposited_energy(0):.2f} GeV into the medium")
print("A zero delta means the deposit was too small, or landed too late, to move the frame at")
print("which the hottest cell drops below T_fo -- the freeze-out grid is dtau =",
      f"{float(P['ideal'].attrs['dtau']):.2f} fm/c, so it is quantised.")

## 8. What to take from this

- The difference between the two datasets in each file **is** the wake: both legs are the same
  solver on the same initial condition with the same droplets, and §2 checks that rather than
  assuming it.
- `arr` is **not** `arr_bg + source/S`. The source is injected *into* the evolution, so the
  response spreads well beyond the cells the source touches and keeps propagating after the
  last droplet fires. That is why the file carries two evolutions rather than one evolution and
  a source term.
- Viscosity acts on the wake the way a hydrodynamicist would expect: it damps the amplitude and
  broadens the front. §5 states the expectation before reading the number. On the shipped
  central Au+Au configuration the amplitude ratio comes out near $0.65$ and the front broadens
  by several fm.
- The static-medium Mach angle sits near $23°$ here, against $35°$ for a conformal EoS: the
  lattice equation of state is softer near the transition, so $c_s \approx 0.39$ rather than
  $1/\sqrt3$. Any angle you measure off the maps should exceed it, because the fireball is
  expanding and the transverse flow at the front opens the cone.
- The jet extends the medium's life: $+0.2$ fm/c (ideal) and $+0.3$ fm/c (viscous) here, from
  31 GeV deposited. The effect is real but small, and on a coarser grid or a smaller deposit it
  rounds to zero — which is a statement about the $\Delta\tau$ the freeze-out frame is
  quantised on, not about the physics.

**Caveats.** One event, one $\eta/s$, shear only. The grid is coarse enough that the front
width is quantised by the cell size, so §4's width column is an ordering, not a measurement.
Droplets that deposit after the hydro window are never injected — the run prints how much
energy that loses, and it is worth checking before drawing conclusions from the totals.